1. Import Libraries

In [ ]:
## kernel - ids
"""
Persist Salesforce object field metadata (df_meta) into a SQL Server table.

- Creates the table if it doesn't exist
- Appends new rows OR performs an UPSERT (MERGE) if enabled
- Uses fast_executemany for performance
"""

#Importing Other Libraries
from simple_salesforce import Salesforce,SalesforceLogin,SalesforceAuthenticationFailed,SFType
import pandas as pd
import json
import numpy as np
import time as t
import uuid
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from collections import OrderedDict

from core_functions import *
from sf_source_objects import *
from sf_target_objects import *
#from sf_toTransfer_objects import *
from sf_target_fields import *



In [ ]:
# ----------------------------
# PARAMETERS
# 1) CONFIGURATION
# ----------------------------
p_db_server  = 'Rhombus-SQL-Ser'
p_db  = 'SalesforceMigration'
p_sch = 'ids'
p_tbl_fd = 'sf_object_fd'
p_tbl_pl = 'sf_object_pl'
p_sf_src = 'hubuat'
# p_sf_tgt = 'rhpartial'
# p_sf_tgt = 'rhdata'
#p_sf_tgt = 'rhuat'
p_sf_tgt = 'rhprod'
#p_sf_tgt = 'datasb'

# ----------------------------
# CONSTANTS
# WRITE: APPEND or UPSERT
# ----------------------------
UPSERT_ENABLED = True  # set False to just append/replace
WRITE_CHUNK_SIZE = 10_000

connection_string = (
    f"mssql+pyodbc://@{p_db_server}/{p_db}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&fast_executemany=True"
)


## Configure Logging
This section initializes a logging framework to monitor the ETL workflow’s execution and facilitate debugging by:
Creating an isolated logger named ETL_Salesforce.
Setting the log level to INFO to capture all important messages.
Outputting formatted messages (with timestamps and levels) to the console.
Disabling propagation to prevent duplicate log entrie

In [ ]:
# Configuring logging
import logging
 
# Create a dedicated logger for the ETL process
logger = logging.getLogger('ETL_Salesforce')
logger.setLevel(logging.INFO)
 
# Clear any existing handlers to avoid duplicates
logger.handlers = []  # Remove all handlers from this logger
 
# Create a new StreamHandler with the desired format
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(handler)
 
# Disable propagation to prevent root logger interference
logger.propagate = False

## Establish Connections
This section establishes a connection to the source Salesforce environment using credentials securely fetched from Google Colab’s secrets storage.
Connecting to the Source Salesforce Instance:

In [ ]:
#Connecting To The Source Salesforce Instance
from secrets_ids import secrets
url = secrets.get(p_sf_src, {}).get('site', 'Unknown')
username = secrets.get(p_sf_src, {}).get('username', 'Unknown')
pswd = secrets.get(p_sf_src, {}).get('password', 'Unknown')
token = secrets.get(p_sf_src, {}).get('token', 'Unknown')

src_sf_instance = connect_to_salesforce(url=url, username=username, password=pswd, token=token, domain='test', logger=logger)
 
logger.info(f"Source SF Instance Session ID = {src_sf_instance.session_id}")

To maintain security, credentials are not hardcoded. Instead, they are stored in Google Colab’s Secrets vault. The application dynamically accesses these stored values (src_username, src_password, and src_token) via the userdata.get('secret_name') method. The fetched credentials are passed to the connect_to_salesforce function to authenticate and create a Salesforce client instance.

Connecting to the Destination Salesforce Instance:

In [ ]:
#Connecting To The Destination Salesforce Instance
from secrets_ids import secrets
#import importlib, secrets_ids as secrets
#import importlib
#importlib.reload(secrets)
url_tgt = secrets.get(p_sf_tgt, {}).get('site', 'Unknown')
print(url_tgt)
username_tgt = secrets.get(p_sf_tgt, {}).get('username', 'Unknown')
print(username_tgt)
pswd_tgt = secrets.get(p_sf_tgt, {}).get('password', 'Unknown')
token_tgt = secrets.get(p_sf_tgt, {}).get('token', 'Unknown')

dst_sf_instance = connect_to_salesforce(url=url_tgt, username=username_tgt, password=pswd_tgt, token=token_tgt, domain='login', logger=logger)
 
logger.info(f"Destination SF Instance Session ID = {dst_sf_instance.session_id}")

In [ ]:
print(pd.read_sql("""
    SELECT @@SERVERNAME AS server_name,
           DB_NAME()    AS database_name,
           (SELECT COUNT(*) FROM sys.tables WHERE schema_id = SCHEMA_ID('sfrhd')) AS sfrhd_tables
""", engine))

## Temp run for 1 file for running Diagnose

In [ ]:
import importlib, load_content_documents as lcd
importlib.reload(lcd)

lcd.diagnose_link(dst_sf_instance, engine)

In [ ]:
import importlib, load_content_documents as lcd
importlib.reload(lcd)
sf = dst_sf_instance

lcd.upload_versions(sf, engine, limit=25)
lcd.create_links(sf)
lcd.summary()

In [ ]:
import pandas as pd
base = r"E:\Files_Extract\Salesforce_ContentDocuments"
rd = lambda f: pd.read_csv(f"{base}\\{f}", dtype=str, keep_default_na=False, encoding="utf-8-sig")
d, l = rd("content_documents_prod.csv"), rd("content_document_links_prod.csv")

print(len(d), len(l))                          # expect 88,271 files and 89,127 links
print(d.TargetLoadStatus.value_counts())       # expect only Pending
print(l.TargetLinkStatus.value_counts())       # expect only PendingTargetMapping
print((l.TargetLinkedEntityId != "").sum(), (d.TargetContentVersionId != "").sum())   # expect 0 and 0

## Prod Pilot

In [ ]:
import sys, importlib
sys.path.append(r"E:\\Files_Extract\\Salesforce_ContentDocuments")   # where the .py now lives
import load_content_documents as lcd
importlib.reload(lcd)

sf = dst_sf_instance                      # must be the PRODUCTION connection
print(sf.sf_instance)
assert "sandbox" not in sf.sf_instance, "This is still the sandbox - stop."

# set these AFTER the reload, in every new session
lcd.DOCS_FILE   = "content_documents_prod.csv"
lcd.LINKS_FILE  = "content_document_links_prod.csv"
lcd.TARGET_HOST = sf.sf_instance          # writes are only allowed to this org

lcd.map_target_entities(engine)           # find prod parent Ids in sfrhd
lcd.flag_already_in_target(sf)            # skip files already there from 2024
lcd.check_parents_visible(sf)             # can the migration user see the parents?

In [ ]:
lcd.upload_versions(sf, engine, limit=25)
lcd.create_links(sf)
lcd.summary()

In [ ]:
import urllib
from sqlalchemy import create_engine

conn_str = (r'DRIVER={SQL Server};SERVER=Rhombus-SQL-Ser;'
            r'DATABASE=SalesforceMigration;Trusted_Connection=yes')
params = urllib.parse.quote_plus(conn_str)
engine = create_engine("mssql+pyodbc:///?odbc_connect=%s" % params,
                       fast_executemany=True, isolation_level="AUTOCOMMIT")

# quick check that the sfrhd tables are visible
import pandas as pd
print(pd.read_sql("SELECT TOP 1 [Id], [Metazoa_External_ID__c] FROM [sfrhd].[Case]", engine))

In [ ]:
lcd.map_target_entities(engine)

In [ ]:
lcd.upload_versions(dst_sf_instance, engine, limit=25)

In [ ]:
lcd.create_links(dst_sf_instance)
lcd.summary()

## Data Transformation Phase
This section transforms the extracted Salesforce data and prepares it for integration by:
Setting External_ID__c as the reference column, this is a custom Salesforce field typically used to uniquely identify and cross-reference records with external data sources or across Salesforce orgs.
Creating a new DataFrame df_transformed containing:
Name: copied from the original data.
Phone: copied from the original data, with missing values replaced by an empty string (”) to prevent data quality issues during loading.
External_ID__c: a new column populated by generating a unique UUID for each record, ensuring every row has a distinct identifier for reliable upsert and deduplication operations.
Logging the total number of records and confirming that all values in External_ID__c are unique, which is required for upsert/outbound integration and helps prevent data collisions.
Displaying the first five rows of the transformed DataFrame for review.

## Data Loading Phase
This section loads the transformed data into the destination Salesforce instance. The following steps are taken:
Defines the list of fields [‘Name‘, ‘Phone‘, ‘External_ID__c‘] which will be included in the upsert operation. 
Calculates the batch size for the bulk operation as the minimum of 1000 and the total number of records in df_transformed. 
Calls the load_data function with parameters:
Destination Salesforce instance connection.
Transformed data DataFrame.
Target object Account.
Upsert reference field External_ID__c.
Fields to include.
Batch size for efficiency and API compliance.
The results of the load operation are collected into the df_load_results DataFrame, which typically contains success/failure info for each record processed.
Displays the first five rows of the load results for quick inspection and verification.

## Phase 2 - Get New Salesforce IDs and update objects with reference fields

## Phase 2 - User - Internal, not associated with Account and Contact
### Only update recods with matched Metazoa_External_Id__c

In [ ]:
# User object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_User-ALL]
;"""

In [ ]:
from sf_target_fields import *

# User object in RA
# ls_tgt_objFields = [
#     'Id'
#     ,'Metazoa_External_ID__c'
# ]
ls_target_fields = USER_TARGET_FIELDS

ls_tgt_objFields = [x for x in ls_target_fields if x not in ('OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate')]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total User Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='User', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000,logger=logger,mode='UPSERT')
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='User', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000,logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - PermissionSet Object

In [ ]:
# Account object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_PermissionSet-ALL]
;"""

In [ ]:
from sf_target_fields import *

ls_tgt_objFields = PERMISSIONSET_TARGET_FIELDS

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total PermissionSet Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='PermissionSet', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - PermissionSetAssignment Object (for Community Users)

In [ ]:
# Account object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_PermissionSetAssignment-ALL]
;"""

In [ ]:
from sf_target_fields import *

ls_tgt_objFields = [
'ExpirationDate'
# ,'IsActive'
# ,'IsRevoked'
,'PermissionSetId'
,'PermissionSetGroupId'
,'AssigneeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total PermissionSetAssignment Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='PermissionSetAssignment', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Account Object

In [ ]:
# Account object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Account-ALL]
;"""

In [ ]:
from sf_target_fields import *

ls_tgt_objFields = ACCOUNT_TARGET_FIELDS

In [ ]:
ls_tgt_objFields

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Account Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Account', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Contact

In [ ]:
# Contact object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Contact-ALL]
;"""

In [ ]:
# Contact object in RA
from sf_target_fields import *

ls_tgt_objFields = CONTACT_TARGET_FIELDS

# ls_tgt_objFields = [
#     'Id',
#     'Metazoa_External_ID__c'
# ]

In [ ]:
ls_tgt_objFields

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Contact Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Contact', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Contact', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - User - Community, associated with Account and Contact

In [ ]:
# Opportunity object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_User-Community]
;"""

In [ ]:
from sf_target_fields import *

ls_tgt_objFields= USER_TARGET_FIELDS

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total User Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='User', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - CSO__c

In [ ]:
# Opportunity object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CSO-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Activity_Achieved__c'
,'Activity_Exceed__c'
,'Activity_Partial_achieved__c'
,'Area__c'
,'Business_City__c'
,'Business_State_Province__c'
,'Business_Street__c'
,'Business_Zip_Postal_Code__c'
,'CSOStatus__c'
,'Email__c'
,'First_Name__c'
,'FTE__c'
,'Last_Name__c'
,'Mobile__c'
# ,'Name'
,'Preferred_Name__c'
,'Recovery_Charge_Rate__c'
,'Responsibility__c'
,'Salutation__c'
,'Team_Leader__c'
,'XPLAN_ID__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
logger.info(f"Total CSO__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CSO__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - PSA__c

In [ ]:
# Opportunity object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_PSA-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Average_AQR_Score__c'
,'Comments__c'
,'EDP_External_Id__c'
,'Is_Active__c'
# ,'Name'
,'Number_Of_Open_Tasks__c'
,'On_Watch_List_Manually__c'
,'Review_Date__c'
,'Review_Score__c'
,'Status__c'
,'Unanswered_Questions__c'
,'Upcoming_Task_Due_Date__c'
,'Practice_Name__c'
,'Secondary_Contact__c'
,'Licensee_Supervisor_Name__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total PSA__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='PSA__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Special_Conditions__c

In [ ]:
# Opportunity object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Special_Conditions-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Application_Number__c'
,'Comments__c'
,'Due_Date__c'
,'EDP_External_Id__c'
,'End_Date__c'
# ,'Name'
,'Special_Conditions_Description__c'
,'Special_Conditions_Type__c'
,'Start_Date__c'
,'Status__c'
,'Adviser__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Special_Conditions__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Special_Conditions__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Target__c

In [ ]:
# Opportunity object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Target-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Actual__c'
,'End_Date__c'
,'Name'
,'Points_Achieved_SFG__c'
,'Shading__c'
,'Shading_Comments__c'
,'Start_date__c'
,'Target__c'
,'User__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Target__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Target__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Tracking__c

In [ ]:
# Opportunity object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Tracking-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Comments__c'
,'Date__c'
,'End_Date__c'
,'Hrs__c'
# ,'Name'
,'Start_date__c'
,'TrackingUniqueKey__c'
,'Type__c'
,'RecordTypeId'
,'Target__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Tracking__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Tracking__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Campaign

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Campaign-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'ActualCost'
,'AmountAllOpportunities'
,'AmountWonOpportunities'
,'BudgetedCost'
,'Description'
,'Duration__c'
,'EndDate'
,'ExpectedResponse'
,'ExpectedRevenue'
,'IsActive'
,'Location__c'
,'Name'
,'NPS__c'
,'NumberOfContacts'
,'NumberOfConvertedLeads'
,'NumberOfLeads'
,'NumberOfOpportunities'
,'NumberOfResponses'
,'NumberOfWonOpportunities'
,'NumberSent'
,'Reason_for_training__c'
,'StartDate'
,'Status'
,'Topic_Area__c'
,'Total_Bridges_Attendees__c'
,'Total_Consultum_Attendees__c'
,'Total_FSP_Attendees__c'
,'Total_GPG_Attendees__c'
,'Total_Internal_Attendees__c'
,'Total_Lonsdale_Attendees__c'
,'Total_M3_Attendees__c'
,'Total_PY_Attendees__c'
,'Total_RI_Attendees__c'
,'Total_Shadforth_Attendees__c'
,'Total_TenFifty_Attendees__c'
,'Training_mode_Method_of_contact__c'
,'Training_Start_Date__c'
,'Type'
,'ParentId'
,'CampaignMemberRecordTypeId'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Campaign Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Campaign', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Breach_Reporting__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - CampaignMember

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CampaignMember-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
# 'Metazoa_External_ID__c'
# 'City'
# ,'CompanyOrAccount'
# ,'Country'
# ,'Description'
# ,'DoNotCall'
# ,'Email'
# ,'Fax'
# ,'FirstName'
# ,'FirstRespondedDate'
# ,'HasOptedOutOfEmail'
# ,'HasOptedOutOfFax'
# ,'HasResponded'
# ,'LastName'
# ,'LeadSource'
'Licensee_text__c'
# ,'MobilePhone'
# ,'Name'
# ,'Phone'
# ,'PostalCode'
# ,'Salutation'
# ,'State'
,'Status'
# ,'Street'
# ,'Title'
# ,'Type'
# ,'LeadOrContactId'
,'CampaignId'
,'ContactId'
# ,'LeadOrContactOwnerId'
,'LeadId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total CampaignMember Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CampaignMember', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Breach_Reporting__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Opportunity

In [ ]:
# Opportunity object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Opportunity-ALL]
;"""

In [ ]:
from sf_target_fields import *

ls_tgt_objFields = OPPORTUNITY_TARGET_FIELDS

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Opportunity Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - LookupName__c

In [ ]:
# Case object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_LookupName-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Action_Required__c'
,'Issue_Identified__c'
# ,'metazoa_external_id__c'
,'Name'
,'Remediation_Timeframe__c'
,'Security_Measure__c'
,'SecurityCode__c'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total LookupName__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='LookupName__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Product2

In [ ]:
# Case object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Product2-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Description'
,'DisplayUrl'
,'EDP_External_Id__c'
,'ExternalId'
,'Family'
,'IsActive'
# ,'IsArchived'
,'Licensee_Groups_Permitted__c'
,'Name'
,'ProductCode'
,'QuantityUnitOfMeasure'
,'StockKeepingUnit'
,'ExternalDataSourceId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Product2 Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Product2', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Registered_Training_Organisation__c

In [ ]:
# Case object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Registered_Training_Organisation-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'EDP_External_Id__c'
,'Name'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Registered_Training_Organisation__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Registered_Training_Organisation__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - BID_Case__c

In [ ]:
# Case object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_BID_Case-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Adviser_ph_call_completed__c'
,'Area_of_advice__c'
,'Call_invoiced_to_client__c'
,'Case_issue__c'
,'Case_Status__c'
,'Complexity_of_Advice__c'
,'Customer_1_First_Name__c'
,'Customer_1_Surname__c'
,'Customer_2_First_Name__c'
,'Customer_2_Surname__c'
,'Date_BID_review_completed__c'
,'Date_call_completed__c'
,'Date_case_due__c'
,'Date_of_BID_review_received__c'
,'Date_scheduled_audit__c'
,'Days_overdue__c'
,'Initial_triage_outcome__c'
,'Integration_ID__c'
,'Invoiced_to_the_client__c'
,'IsActive__c'
,'Name'
,'No_of_new_documents_received_01__c'
,'No_of_new_documents_received_02__c'
,'Outcome_of_BID_review__c'
,'Resub01_date_completed__c'
,'Resub01_date_received__c'
,'Resub01_invoiced_to_client__c'
,'Resub01_Outcome_of_BID_review__c'
,'Resub02_date_completed__c'
,'Resub02_date_received__c'
,'Resub02_invoiced_to_client__c'
,'Resub02_Outcome_of_Bid_review__c'
,'Resubmission_01_Case_Status__c'
,'Resubmission_02_Case_Status__c'
,'SubmissionNumber__c'
,'Adviser__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total BID_Case__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='BID_Case__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Case

In [ ]:
# Case object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Case-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
# ,'CaseNumber'     -- not updatable
,'ABN_ACN_Check__c'
,'ABN_ACN_Check_Date__c'
,'AccountId'
,'Additional_Directorships_Check__c'
,'Additional_Directorships_Check_Date__c'
,'ADG_ANZ_child__c'
# # ,'Age_without_Deed_Pending_Duration__c'       -- caLculated
# # ,'Aged_care__c'       -- obsent
# # ,'Aged_care_Completion_Date__c'       -- obsent
,'Amount_Paid_to_Client__c'
,'Approval_Comments__c'
,'Approvals_Professional_Standard__c'
,'APRA_Check__c'
,'APRA_Check_Date__c'
,'AR_Register_Check__c'
,'AR_Register_Check_Date__c'
,'Area_Responsible__c'
,'ASIC_Check__c'
,'ASIC_Check_Date__c'
,'ASIC_EU_Check__c'
,'ASIC_EU_Check_Date__c'
,'AssetId'
,'Banned_Disqualified_Check__c'
,'Banned_Disqualified_Check_Date__c'
,'Business_insurance__c'
,'Business_insurance_Completion_Date__c'
,'BusinessHoursId'
# # ,'Case_Age__c'    -- calculated
# # ,'Case_Closed_Date__c'    -- calculated
,'Case_Managed_by__c'
,'Claimed_Amount__c'
,'Client_1_First_Name__c'
,'Client_1_Last_Name__c'
,'Client_2_First_Name__c'
,'Client_2_Last_Name__c'
,'Client_3_First_Name__c'
,'Client_3_Last_Name__c'
,'Client_Name__c'
,'ClientID_Xplan__c'
,'Closed_by_EDR_Scheme__c'
# ,'ClosedDate'     -- not updatable
,'Comments'
,'Complainant__c'
,'Complainant_age__c'
,'Complainant_Claim_Amount__c'
,'Complainant_gender__c'
,'Complainant_postcode__c'
,'Complainant_Type__c'
,'Complaint_Allegation__c'
,'Complaint_channel__c'
,'Complaint_Closure_Checklist_completed__c'
,'Complaint_issue__c'
,'Complaint_issue_14_b__c'
,'Complaint_issue_14_c__c'
,'Complaint_outcome__c'
,'Complaint_outcome_15_b__c'
,'Complaint_outcome_15_c__c'
# ,'Complaint_Received_Date__c' -- calculated
,'Conciliation_Date__c'
,'Confirmation_Provided__c'
,'Consequence_Management_Form_required__c'
# ,'ContactEmail'     -- not updatable
# ,'ContactFax'     -- not updatable
,'ContactId'
# ,'ContactMobile'     -- not updatable
# ,'ContactPhone'     -- not updatable
,'COR_Closure_request__c'
,'COR_ID__c'
,'COR_Logged_date__c'
# ,'CreatedById'     -- not updatable
# ,'CreatedDate'     -- not updatable
,'Credit_Bankruptcy__c'
,'Credit_Bankruptcy_Check_Date__c'
,'Credit_Rep_Register_Check__c'
,'Credit_Rep_Register_Check_Date__c'
,'CSA_Extension_Approval_Sent_To_Adviser__c'
,'CSA_extension_end_date__c'
# ,'Current_Adviser__c' -- calculated
,'Date_Closed__c'
,'Date_PI_insurer_notified__c'
,'Date_Received_by_Company_ADG__c'
,'Date_Received_by_Complaints_Team__c'
,'DDO_Product__c'
,'DDO_Reportable__c'
,'Description'
,'Determination_Date__c'
# ,'Direct_Shares__c'   -- obsent
# ,'Direct_Shares_Completion_Date__c'   -- obsent
# ,'Due_Date__c'    -- calculated
,'EDP_External_Id__c'
,'EDR_Scheme__c'
,'Enforceable_Undertaking_Check__c'
,'Enforceable_Undertaking_Check_Date__c'
,'EntitlementId'
,'Escalation_Case__c'
,'Extension_Letter_sent__c'
,'External_Reference_Number__c'
,'FAR_Check__c'
,'FAR_Check_Date__c'
,'Fee_Reimbursement_Program_Complaint__c'
,'Files_Available__c'
,'Follow_up_comment__c'
,'Follow_up_Date__c'
# ,'Gearing__c'     -- obsent
# ,'Gearing_Completion_Date__c'     -- absent
,'Has_a_reportable_situation_occurred__c'
,'Insurance__c'
# ,'Internal_Raiser__c'    -- need user lookup
,'Internet_And_Social_Media_Check__c'
,'Internet_And_Social_Media_Check_Date__c'
,'Investment__c'
,'Is_the_complaint_about_the_authorised_re__c'
# ,'IsClosed'     -- not updatable
# ,'IsClosedOnCreate'
# ,'IsDeleted'     -- not updatable
,'IsEscalated'
,'IsStopped'
,'Language'
# ,'LastModifiedById'     -- not updatable
# ,'LastModifiedDate'     -- not updatable
# ,'LastReferencedDate'     -- not updatable
# ,'LastViewedDate'     -- not updatable
# ,'Licensee_logo__c'   -- calculated
# ,'Licensee_Name__c'   -- calculated
# ,'Licensee_Supervisor__c'
,'Licensee_Supervisor_Name__c'
,'Loss_Incurred_by_which_BSBCC__c'
# ,'Margin_Lending__c'      -- obsent
# ,'Margin_Lending_Completion_Date__c'      -- obsent
# ,'MasterRecordId'     -- not updatable
,'Method_Received__c'
# ,'MilestoneStatus'     -- not updatable
,'Monetary_remedy__c'
,'Non_Authorised_Director__c'
,'Non_Authorised_Director_Date__c'
,'Notified_to_Escalation_Teams__c'
,'Number_and_Type_13_a__c'
,'Number_and_Type_13_b__c'
,'Number_and_Type_13_c__c'
,'Number_of_Open_Tasks__c'
,'Onboarding_Offboarding__c'
,'Ongoing_Service_Client__c'
,'Origin'
,'Other_Product__c'
# ,'Other_Qualifications__c'    -- obsent
# ,'OwnerId'     -- not updatable
,'ParentId'
,'Payment_Made__c'
,'Pension__c'
,'PEP__c'
,'PEP_Client__c'
# ,'PI_insured_amount_over_20k__c'  -- calculated
,'PI_insurer_indemnity_status__c'
,'PI_Insurer_Notified__c'
,'PI_Reference__c'
,'Platform__c'
,'Platform_Other__c'
,'Practice_Name__c'
,'Priority'
,'Privacy_Related__c'
,'Product__c'
,'Product_Not_Applicable__c'
,'Product_or_service_13_a__c'
,'Product_or_service_13_b__c'
,'Product_or_service_13_c__c'
,'Product_or_service_category__c'
,'Product_or_service_category_FinAd__c'
,'Product_or_service_category_LIL__c'
,'Product_or_service_category_Not_Prod__c'
,'Product_or_service_category_SupA__c'
,'Product_or_service_category_TraTS__c'
,'Product_or_service_number_and_type__c'
,'Product_or_service_number_and_type_FinAd__c'
,'Product_or_service_number_and_type_LIL__c'
,'Product_or_service_number_and_type_NotP__c'
,'Product_or_service_number_and_type_SupA__c'
,'Product_or_service_number_and_type_TraTS__c'
,'Product_Other__c'
,'Product_Type__c'
,'ProductId'
,'Qualifications__c'
,'Reason'
,'Recommendation_Date__c'
# ,'Record_Type__c' -- calculated
,'RecordTypeId'
,'Registration_Due__c'
,'Registration_Received__c'
,'Registration_Sent__c'
,'Reportable_as_an_incident__c'
,'Resolved_in_5_Business_Days_Date_Closed__c'
,'Resolved_in_Favour__c'
,'Response__c'
,'Response_Letter_Approved_by_Licensee__c'
,'Review_Date__c'
# ,'Reviewer__c'    -- need user lookup
,'RG146_Review__c'
,'RG146_Review_Date__c'
,'RI_Technical_Knowledge_Test__c'
,'RI_Technical_Knowledge_Test_Date__c'
,'Root_Cause__c'
,'Sanctions_Check__c'
,'Sanctions_Check_Date__c'
# ,'SlaExitDate'     -- not updatable
,'SlaStartDate'
,'SourceId'
# ,'Specialist_Accreditation__c'    -- obsent
,'Stage__c'
# ,'State_of_Adviser__c'    -- calculated
,'State_of_Client__c'
,'Status'
# ,'Status_ASIC__c'     -- calculated
,'Status_of_Apporval__c'
# ,'StopStartDate'     -- not updatable
# ,'Sub_Category_Subject_Area__c'     -- INVALID_OR_NULL_FOR_RESTRICTED_PICKLIST
,'Sub_Product_Type__c'
,'Sub_Root_Cause__c'
# ,'Sub_Sub_Category__c'     -- INVALID_OR_NULL_FOR_RESTRICTED_PICKLIST
,'Subcategory_13_a__c'
,'Subcategory_13_b__c'
,'Subcategory_13_c__c'
,'Subject'
# ,'Subject_Area__c'     -- INVALID_OR_NULL_FOR_RESTRICTED_PICKLIST
,'Substantive_Response_Sent__c'
,'Super__c'
,'SuppliedCompany'
,'SuppliedName'
,'SuppliedPhone'
# ,'SystemModstamp'     -- not updatable
,'Tax_Practitioner_Board__c'
,'Technical_Knowledge_Date__c'
,'Technical_Knowledge_Test__c'
,'TPB_Check_Date__c'
,'Trust_Deed_Checks__c'
,'Trust_Deed_Checks_Date__c'
,'Type'
,'Upcoming_Task_Due_Date__c'
# ,'Work_Item_Number__c'    -- calculated
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Case Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Case_Snapshot__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Case_Snapshot-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Case_Type__c'
,'EDP_External_Id__c'
# ,'Name'
,'Schedule_Date__c'
,'Schedule_Type__c'
,'Status__c'
,'Case__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Case_Snapshot__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case_Snapshot__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Assurance_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Assurance_Event__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Assurance_Event-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Closed_Date__c'
,'Comments__c'
,'Event_Record_Sub_Type__c'
,'Event_Record_Type__c'
,'Event_Referral_Date__c'
,'Event_Scope__c'
,'Event_Scope_Comment__c'
,'Event_Status__c'
# ,'Name'
,'Number_Of_Client_Files__c'
,'Number_Of_Unique_Advisers__c'
,'Review_Count__c'
,'Related_Incident__c'
,'RecordTypeId'
,'Event_Lead__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Assurance_Event__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Assurance_Event__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Assurance_Review__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Assurance_Review-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Actions_from_Vetting_Result__c'
,'Advice_Document_Type__c'
,'Advice_Document_Type_AdviserONE__c'
,'Advice_Risk_Escalation_Comments__c'
,'Advice_Risk_Escalation_Recommendations__c'
,'All_Response_Yes__c'
,'Any_Outstanding_Remediation__c'
,'Appeal_Closed_Date__c'
,'Appeal_Received_Date__c'
,'Appeal_Status__c'
,'Appointment_Date__c'
,'Appointment_Location__c'
,'Appointment_Time__c'
,'Are_all_Documents_Received__c'
,'Areas_of_Advice_for_Vetting__c'
,'ASN_Score__c'
,'Atleast_Two_Potential_Repeat_Offence_s__c'
,'Client_Name__c'
,'ClientID_Xplan__c'
,'Closed_Date__c'
,'Comment__c'
,'Comment1__c'
,'Comment2__c'
,'Comment3__c'
# ,'Comment4__c'    -- not exists
,'Comments_for_Targeted_Review__c'
,'Comments_on_Repeat_Offence_s__c'
,'Continue__c'
# ,'CreatedDate'
,'Current_Score__c'
,'Data_Source_For_Client_File_Selection__c'
# ,'Debrief_Attendees__c'   -- not exists
,'Debrief_Date__c'
,'Debrief_Minutes__c'
,'Debrief_Notes__c'
# ,'Debrief_Time__c'        -- not exists
,'Document_Received_Date__c'
,'Draft_Report_Sent_Date__c'
,'Draft_Score__c'
,'EDP_External_Id__c'
,'Escalate_to_Advice_Risk__c'
,'Final_Report_Sent_Date__c'
,'Final_Score_after_Appeal__c'
,'Final_Score_before_Appeal__c'
,'First_Report_Sent_Date__c'
# ,'Future_Review_Created__c'       -- not exists
,'General_Advice_Score_Impact__c'
,'Guidance_for_General_Advice_Overview__c'
,'Guidance_for_Potential_Compliance_Issues__c'
,'Guidance_for_Potential_Repeat_Offences__c'
,'Guidance_for_Targeted_Review_Suggestion__c'
,'Hidden_in_AdviserONE__c'
,'Initial_Owner_Assignment_Status__c'
,'Interim_or_Final_Report_Modified_on_Date__c'
,'Interim_Report_Sent_Date__c'
# ,'Is_Review_in_Flight__c'     -- not exists
,'Is_Reviewer_Assigned__c'
,'Is_Root_Cause_Analysis_Completed__c'
,'Is_targeted_review_recommended__c'
,'Is_this_a_No_report_review__c'
# ,'IsDeleted'
# ,'Last_Draft_Report_Sent_Date__c'     -- not exists
,'Last_Review_Client_File_Xplan_Id__c'
,'Last_Review_Period__c'
# ,'LastActivityDate'
# ,'LastModifiedDate'
# ,'LastReferencedDate'
# ,'LastViewedDate'
,'Licensee_Name__c'
,'MandatoryQuestionsFail__c'
# ,'Name'
,'Number_of_High_Issues__c'
,'Number_of_Improvement_Options__c'
,'Number_of_Low_Issues__c'
,'Number_of_Medium_Issues__c'
,'Number_Of_Open_Tasks__c'
,'Number_of_Time_Critical_Advice__c'
,'Number_of_Total_Tasks__c'
,'On_Hold__c'
,'One_Client_File_Reviewed__c'
,'Oustanding_Remediation_Actions__c'
,'Overview_Completion_Status__c'
,'Overview_Questionnaire_Status__c'
,'Partner_AFSL__c'
# ,'Picked_For_Accredited_QA__c'        -- not exists
# ,'Picked_For_Standard_QA__c'      -- not exists
,'Pre_review_Completion_Status__c'
,'Pre_Review_Questionnaire_Status__c'
,'Priority__c'
# ,'QA_Done__c'     -- not exists
,'QA_Done_Draft__c'
,'QA_Done_Final__c'
,'QA_Done_Interim__c'
,'QA_Task_Status__c'
,'Re_submission_Status__c'
,'Reason_for_Cancellation__c'
,'Reason_for_No_Advice_Review__c'
,'Reason_for_Review__c'
,'Reason_for_Review_AdviserONE__c'
,'Reason_for_Review_Date_Change__c'
,'Received_Date__c'
,'Recommendation1__c'
,'Recommendation2__c'
,'Recommendation3__c'
,'Recommendations_for_Repeat_Offence_s__c'
,'Recommendations_for_Targeted_Review__c'
,'Recommendations_to_Continue_Doing__c'
,'Recommendations_to_Start_Doing__c'
,'Recommendations_to_Stop_Doing__c'
,'Regulatory_Compliance_Score_Impact__c'
,'Remedial_Actions__c'
,'Remediation_Due_Date__c'
,'Replacement_of_product__c'
,'Report_Finalized_Date__c'
,'Report_Type__c'
,'Require_Vetting__c'
,'Response_Days__c'
,'Response_Due_Date__c'
,'Response1__c'
,'Response2__c'
,'Response3__c'
,'Response4__c'
,'Response5__c'
,'Review_Closed_Date__c'
,'Review_Date__c'
,'Review_Due_Date__c'
,'Review_Period__c'
,'Review_Status__c'
,'Review_Type_Options__c'
,'Scheduling_Frequency__c'
,'Should_the_adviser_be_put_on_vetting__c'
,'Stage__c'
,'Start__c'
,'Stop__c'
,'Sub_Stage__c'
# ,'Supress_Advice_Assurance_Score__c'      -- not exists
# ,'SystemModstamp'
,'Three_or_More_Client_Files_Reviewed__c'
,'Total_Number_of_Issues__c'
,'Two_Client_Files_Reviewed__c'
,'Unanswered_Overview_Questions__c'
,'Vetting__c'
,'Vetting_Commencement_Date__c'
,'Vetting_Report_Text__c'
,'Vetting_Result__c'
,'Workflow_Step__c'
# references
,'Practice__c'
,'Assurance_Event__c'
,'Cloned_Targeted_Review__c'
,'Original_Request_Number__c'
,'Resubmission_case__c'
,'Risk_Forum__c'
,'Adviser__c'
,'Adviser_AdviserONE__c'
,'RecordTypeId'
,'Assurance_Reviewer__c'
,'Licensee_Supervisor_Name__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Assurance_Review__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Assurance_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=100, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Assurance_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Client_Service_Model_Implementation__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Client_Service_Model_Implementation-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'ACR_Go_Live__c'
,'ACR_Initiative_Comments__c'
,'ACR_Question_1__c'
,'ACR_Question_2__c'
,'ACR_Question_3__c'
,'ACR_Question_4__c'
,'ACR_Question_5__c'
,'Additional_Support__c'
,'Advice_Case_Framework_Comments__c'
,'Answer_1__c'
,'Answer_2__c'
,'Answer_3__c'
,'Answer_4__c'
,'Answer_5__c'
,'CKRI_A9__c'
,'CKRI_NotUmlaut_Comments__c'
,'CKRI_Q7__c'
,'CKRI_Q8__c'
,'CKRI_Q9__c'
,'CombinationBanner__c'
,'Comments_for_Combination__c'
,'Completion_Date__c'
,'CRKI_A1__c'
,'CRKI_A10__c'
,'CRKI_A11__c'
# ,'CRKI_A2_Number_of_Years__c'       -- invalid batch
,'CRKI_A3__c'
,'CRKI_A4__c'
,'CRKI_A5_c__c'
,'CRKI_A6__c'
,'CRKI_A7__c'
,'CRKI_Answer_0__c'
,'CRKI_Comments_A1__c'
,'CRKI_Q1__c'
,'CRKI_Q11__c'
,'CRKI_Q11_Comments__c'
,'CRKI_Q2__c'
,'CRKI_Q3__c'
,'CRKI_Q4__c'
,'CRKI_Q5__c'
,'CRKI_Q6__c'
,'CRKI_Question_0__c'
,'Cyber_Assessment_Due_Date__c'
,'Cyber_Assessment_Required__c'
,'Cyber_Insurance_Expiry_Date__c'
,'CyberAssessment_Completed_Date__c'
,'CyberSecurity_Assessment_Comments__c'
# ,'CyberSecurity_Assessment_Due_Date__c'   -- not exists
,'Date_of_2nd_Assessment__c'
,'Date_of_Assessment__c'
,'Date_of_Engagement__c'
,'Date_of_Engagement_2__c'
,'Date_of_Engagement_3__c'
,'Date_of_outcome_report__c'
,'Decision_Made_on_New_Licensee__c'
,'Detect_Score__c'
,'Electronic_Storage_Requirement_Comments__c'
,'Engaged_Advisers2__c'
,'Estimated_Completion_Date_of_Pre_Work__c'
,'Estimated_Final_Completion_Date__c'
,'Identify_Score__c'
,'Impacted_by_Advice_2_0__c'
,'LicenseeSupervisorEmail__c'
,'Manage_Client_Reviews_Comments__c'
,'Minimum_Data_Requirement_Comments__c'
# ,'Name'       -- not allowed
,'Number_of_Client_Files__c'
,'Number_of_Years_Digital__c'
,'Number_of_Years_Paper__c'
,'Number_of_Years_XPLAN__c'
,'Overall_Comments__c'
,'Overall_Score__c'
,'Practice_contact__c'
,'PracticeEmail__c'
,'Pricing__c'
,'Protect_Score__c'
,'Question_1__c'
,'Question_1_Comments__c'
,'Question_10__c'
,'Question_2__c'
,'Question_2_Comments__c'
,'Question_3__c'
,'Question_3_Comments__c'
,'Question_4__c'
,'Question_4_Comments__c'
,'Recover_Score__c'
,'Respond_Score__c'
,'Risk_Insurance_Only__c'
,'Services__c'
,'Stage_1_Status__c'
,'Stage_2_Status__c'
,'Status__c'
,'System_Xplan__c'
,'Transition_Process__c'
,'What_CRM_or_server_system_does_the_pract__c'
,'Practice__c'
,'Primary_Practice__c'
,'Advisers_Engaged__c'
,'RecordTypeId'
,'Owner__c'
,'OwnerBackBookAttestation__c'
,'OwnerProjectEngagement__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Client_Service_Model_Implementation__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Incident_File_Review__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Incident_File_Review-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'A1__c'
,'A2__c'
,'A3__c'
,'A4__c'
,'A5__c'
,'Account_Number__c'
,'Actions_Completed_to_Date__c'
,'Application_Date__c'
,'ASIC_Notification__c'
,'Client_Name__c'
,'Close_Date__c'
,'Comments__c'
,'COR_ID__c'
,'COR_Logged_Date__c'
,'COR_Status__c'
,'Date_Closed_in_COR__c'
,'Date_Occurred__c'
,'Date_of_ASIC_Notification__c'
,'Date_of_Governance_Forum_Escalation__c'
,'Date_of_Payment__c'
,'Date_of_Referral__c'
,'Date_PI_Insurer_Notified__c'
,'Discovery_Date__c'
,'EDP_External_Id__c'
,'Event_Type__c'
,'Fact_Find_Date__c'
,'Fault_Area__c'
,'Hidden_in_AdviserONE__c'
,'Incident_Category_Field__c'
,'Incident_Description__c'
,'Incident_Title__c'
,'Incident_Type__c'
,'Is_Service_Arrangement_Documented__c'
,'Is_there_a_fee_discrepancy__c'
,'Is_there_client_authorisation_evidence__c'
,'Is_there_service_delivery_evidence__c'
,'Is_this_a_Major_Incident__c'
,'Licensee__c'
,'Locked_in_AdviserONE__c'
,'Logged_Date__c'
,'Mark_as_Confidential__c'
# ,'Name'       -- not allowed
,'Number_of_Clients_Impacted__c'
,'Number_of_High_issues__c'
,'Number_of_Low_issues__c'
,'Number_of_Medium_issues__c'
,'Number_of_Open_Incidents__c'
,'Partner_AFSL__c'
,'PI_Insurer_Indemnity_Status__c'
,'PI_Insurer_Notified__c'
,'PI_Reference__c'
,'Potential_Loss_in_terms__c'
,'Prior_Owner__c'
,'Product_Provider__c'
,'Professional_Indemnity_Amount__c'
,'Related_to_a_Service_Arrangement__c'
,'Relevant_Service_Period_End_Date__c'
,'Relevant_Service_Period_Start_Date__c'
,'Remediation_Due_Date__c'
,'Remediation_Provided_Date__c'
,'Report_Sent_Date__c'
,'REV__c'
,'Root_Cause__c'
,'Service_Arrangement_Type__c'
,'SOA_Date__c'
,'SOA_Produced_By__c'
,'Source__c'
,'Stage__c'
,'Submitted_Date__c'
,'Total_Number_of_Incidents__c'
,'Total_number_of_issues__c'
,'Wave_Internal__c'
,'Xplan_ID__c'
,'Practice__c'
,'Assurance_Review__c'
,'Adviser__c'
,'Adviser_AdviserONE__c'
,'Area_of_Advice__c'
,'AssignedTo__c'
,'Licensee_Supervisor__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Incident_File_Review__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Incident_File_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=1000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Incident_File_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - QA__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_QA-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Debrief__c'
,'Debrief_Notes__c'
,'Draft_Report__c'
,'Final_Report__c'
,'Interim_Report__c'
# ,'Name'
,'Other_Discussion_Points__c'
,'QA_Completed_On__c'
,'QA_Date__c'
,'QA_Outcome__c'
,'QA_Status__c'
,'QA_Type__c'
,'Questionnaire_Launched_On__c'
,'Questionnaire_Status__c'
,'Review_Questionnaire__c'
,'Scope_Strategies__c'
,'Practice__c'
,'Review__c'
,'Adviser__c'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total QA__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='QA__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - QA_Assignment_History__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_QA_Assignment_Hisotry-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Name'
,'QA_Type__c'
,'Assurance_Review__c'
,'QA_Officer__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total QA_Assignment_Hisotry__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='QA_Assignment_Hisotry__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Accreditations__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Accreditations-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Document_Received_Date__c'
,'EDP_External_Id__c'
,'Kaplan_Training_Date__c'
# ,'Name'
,'Product_Family__c'
,'Revoke_Date__c'
,'Start_Date__c'
,'Adviser__c'
,'AccreditationType__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Accreditations__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Accreditations__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Paraplanning__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Paraplanning-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Additional_Hours__c'
,'Additional_Hours_new__c'
,'Additional_QDC_Time_Taken__c'
,'Additional_time_taken__c'
,'Amount__c'
,'Answer__c'
,'Approval_Sought_from_PDM__c'
,'Areas_for_Improvement__c'
,'Areas_of_Advice__c'
,'Assigned_QDC_Date__c'
,'Assigned_Review_Date__c'
,'Assigned_Review_Date_aft_2_3_biz_days__c'
,'Assigned_SOA_due_date__c'
,'Assigned_SOA_due_date_Adjusted__c'
,'Client_name__c'
,'Client_Type__c'
,'Comment__c'
,'Comments_Completion_Notes__c'
,'Completed_Date__c'
,'Cost_per_extra_hour__c'
,'CSA_End_Date__c'
,'Data_Entry_Hours__c'
,'Data_Entry_Hours_new__c'
,'Data_Migration_Status__c'
,'Date_Advice_Request_Received__c'
,'Date_Outsourced__c'
,'Date_time_pending_OUT_1__c'
,'Date_time_pending_OUT_2__c'
,'Date_time_pending_OUT_3__c'
,'Date_time_resumed_IN_1__c'
,'Date_time_resumed_IN_2__c'
,'Date_time_resumed_IN_3__c'
,'Days_Pending_1__c'
,'Days_Pending_2__c'
,'Days_Pending_3__c'
,'Defined_Purpose_SOA__c'
,'Due_Date__c'
,'Fees_Standard__c'
,'FUM__c'
,'Goals_and_objectives__c'
,'Gross_Turnaround_days__c'
,'GST_rate__c'
,'Hold_1_Days_to_Exclude_from_TAT_WFA__c'
,'Hold_2_Days_to_Exclude_from_TAT_WFA__c'
,'Hold_3_Days_to_Exclude_from_TAT_WFA__c'
,'Inwealth_Charge__c'
,'Key_and_financial_information__c'
,'Last_ClientID_Xplan__c'
,'Licenseetext__c'
,'Meeting_Held_Date__c'
# ,'Name'
,'NBI__c'
,'Non_Working_Days_to_Exclude_from_TAT_WF__c'
,'On_Hold_Comments__c'
,'On_Hold_Email_Send_Date_1__c'
,'On_Hold_Email_Send_Date_2__c'
,'On_Hold_Email_Send_Date_3__c'
,'Ongoing_Fee_Bridges_Only__c'
,'OnHold__c'
,'Paraplanner_Assigned_Date__c'
,'Paraplanner_Quality_Items_Identified__c'
,'Peer_Review_Completed_Date__c'
,'Peer_Review_Date__c'
,'Plan_Type__c'
,'Planlogic_Charge__c'
,'Points_Allocation_New__c'
,'Points_Allocation_QDC__c'
,'Points_Allocation_SFG__c'
,'Points_Allocation_WFA__c'
,'Production__c'
,'Provide_a_brief_explanation__c'
,'QDC_Check_Completion_Date__c'
,'QDC_Check_Outcome__c'
,'QDC_Check_Time_min__c'
,'QDC_Checking_Area__c'
,'Question_1__c'
,'Question_2__c'
,'Quoting_Comments_Notes__c'
,'Reason_for_seeking_advice__c'
,'Rework__c'
,'Scoping_section__c'
,'SOA_Assigned_Date_aft_2_3_Biz_Days__c'
,'SOA_Document_Type__c'
,'SOA_Due_Date_Adjusted__c'
,'SOA_Ready_for_QDC_Check__c'
,'Status__c'
,'Strategy_Notes__c'
,'Submission_quality_items_identified__c'
,'Team_Location__c'
,'Time_critical_email_send_date__c'
,'Total_fee_factor__c'
,'Triage_End_Date__c'
,'Triage_Start_Date__c'
,'Type_of_QDC_Check__c'
,'Upfront_Fee_Bridges_Only__c'
,'Urgency__c'
,'Urgency_Notes__c'
,'Why_the_Difference__c'
,'Workflow_Analyst_Check_Date__c'
,'Workflow_Analyst_Check_Time_min__c'
,'Workflow_Quality_Items_Identified__c'
,'X1_On_Hold_Reason__c'
,'X2_On_Hold_Reason__c'
,'X3_On_Hold_Reason__c'
,'Adviser__c'
,'Associate_Adviser__c'
,'RecordTypeId'
,'Assigned_Outsourcer__c'
,'Assigned_Paraplanner__c'
,'Assigned_QDC__c'
,'Assigned_Reviewer__c'
,'Assigned_Workflow_Analyst__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Paraplanning__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=1000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Question_Responses__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Question_Responses-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Actions_Taken__c'
,'Actual_Loss_in_terms__c'
# ,'Advice_Type__c'     -- not exists
,'Comment__c'
,'Comment_Yes__c'
,'Comments__c'
,'Date_Paid__c'
,'Document_Name__c'
,'EDP_External_Id__c'
,'External_ID__c'
# ,'Feedback__c'        -- not exists
,'Has_Action__c'
,'Incident_Close_Date__c'
,'Incident_Status__c'
,'Key_Area__c'
,'Licensee__c'
,'Mark_as_Confidential__c'
# ,'Name'       -- not allowed
,'Partner_AFSL__c'
,'Pre_Review_and_Overview_Response__c'
,'Prior_Owner__c'
# ,'QA_Rationale__c'        -- not exists
# ,'QA_Response__c'       -- not exists
,'Question_Number__c'
,'Question_Text__c'
# ,'Rationale__c'     -- not exists
,'Report_Breach__c'
,'Requires_Action__c'
,'Response__c'
,'Responsibility__c'
,'Root_Cause__c'
,'Practice__c'
,'Assurance_Review__c'
,'Client_File__c'
,'Adviser__c'
,'Incident_File_Review__c'
,'Question__c'
# ,'QuestionTr__c'      -- not exists
,'PSA__c'
# ,'Quality_Assurance__c'   -- not exists
,'RecordTypeId'
,'Vetting_Result__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Question_Responses__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - A1_Support_Staff_Registrations__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_A1_Support_Staff_Registrations-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Approval_Status__c'
,'Email_Address__c'
,'First_Name__c'
,'Last_Name__c'
# ,'Name'
,'Role__c'
,'Adviser__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total A1_Support_Staff_Registrations__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Id'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='A1_Support_Staff_Registrations__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - AccountContactRelation

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_AccountContactRelation-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Id'
,'Metazoa_External_ID__c'
,'Comments__c'
,'EDP_External_Id__c'
,'Email__c'
,'EndDate'
,'IsActive'
# ,'IsDirect'   -- forbidden field for upsert
,'Relationship_Strength__c'
,'Roles'
,'StartDate'
,'Status__c'
,'AccountId'
,'ContactId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total AccountContactRelation Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Id'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Third_Party_Provider__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Third_Party_Provider-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Email_Address_1__c'
,'Email_Address_2__c'
,'Email_Address_3__c'
# ,'Name'
,'Third_Party_Provider_End_Date__c'
,'Third_Party_Provider_Name__c'
,'Third_Party_Provider_Start_Date__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Third_Party_Provider__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Email_Address_1__c: {df_transformed['Email_Address_1__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Email_Log__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Third_Party_Provider__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=1000, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Question_Group__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Question_Group-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Description__c'
,'EDP_External_Id__c'
,'External_ID__c'
,'Group_Number__c'
,'Name'
,'Questionnaire__c'
,'Section_Title__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Question_Group__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Group__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Master_Questions__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Questions-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Action__c'
,'Actions_Taken__c'
,'Active_From__c'
,'Active_Until__c'
,'Comment__c'
,'Default_Issue_Text_for_Vetting__c'
,'Default_Remedial_Actions_for_Vetting__c'
,'Description__c'
,'Display_Text_For_Reports__c'
,'EDP_External_Id__c'
,'External_ID__c'
,'Impact_on_Score__c'
,'Incident_Remediation_Text__c'
,'Key_Area__c'
# ,'Name'
,'Points__c'
,'Priority__c'
,'Process_Area__c'
,'Question_Number__c'
,'Question_Text__c'
,'Reportable__c'
,'Reportable_For_Reviews__c'
,'Response_Options__c'
,'Rootcause_Options__c'
,'Score__c'
,'Standard__c'
,'Vetting_Action__c'
,'Vetting_Comments__c'
,'Vetting_Mandatory_Question__c'
,'Wave_Internal__c'
,'Question_Group__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Master_Questions__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Master_Questions__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - MasterQuestions_TargetedReviews__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_MasterQuestions_Targetedreviews-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Action__c'
,'Actions_Taken__c'
,'Active_From__c'
,'Active_Until__c'
,'Comment__c'
,'Default_Issue_Text_for_Vetting__c'
,'Default_Remedial_Actions_for_Vetting__c'
,'Description__c'
,'Display_Text_For_Reports__c'
,'EDP_External_Id__c'
,'External_ID__c'
,'Impact_on_Score__c'
,'Incident_Remediation_Text__c'
,'Is_Bank_Question__c'
,'Key_Area__c'
# ,'Name'
,'Points__c'
,'Priority__c'
,'Process_Area__c'
,'Qn_Text_Key__c'
,'Question_Category__c'
,'Question_Number__c'
,'Question_Text__c'
,'Reportable__c'
,'Reportable_For_Reviews__c'
,'Response_Options__c'
,'Rootcause_Options__c'
,'Standard__c'
,'Vetting_Action__c'
,'Vetting_Comments__c'
,'Vetting_Mandatory_Question__c'
,'Wave_Internal__c'
,'Question_Group__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total MasterQuestions_Targetedreviews__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='MasterQuestions_Targetedreviews__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Library_Text__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Library_Text-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Actions_Required__c'
,'Expiry_Date__c'
,'External_ID__c'
,'Findings__c'
# ,'Name'
,'Master_Question__c'
,'Targeted_Assurance_Question_Bank__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Library_Text__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Library_Text__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Product_Questions__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Product_Questions-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'EDP_External_Id__c'
,'ExternalID__c'
# ,'Name'
,'Master_Question__c'
,'Product__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Product_Questions__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Product_Questions__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Master_workshop__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_workshop-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'IsActive__c'
,'Name'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Master_workshop__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Master_workshop__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Master_Module__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Module-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'IsActive__c'
,'Name'
,'Master_workshop__c'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Master_Module__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Master_Module__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Master_Topic__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Topic-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'IsActive__c'
,'Name'
,'Order__c'
,'Master_module__c'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Master_Topic__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Master_Topic__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Master_Item__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Item-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Description__c'
,'IsActive__c'
,'Mandatory__c'
,'Name'
,'Master_Topic__c'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Master_Item__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Master_Item__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Workshop__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Workshop-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
# ,'Name'
,'Type_of_Workshop__c'
,'UniqueMasterWorkshopAA__c'
,'Workshop_Attended_Date__c'
,'Adviser_Academy__c'
,'Workshop__c'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Workshop__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Workshop__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Module__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Module-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Actual_Complete_Date__c'
,'Actual_Start_Date__c'
,'Comments__c'
,'Completion__c'
,'Description__c'
,'Implementing__c'
# ,'Name'
,'Planned_Complete_Date__c'
,'Planned_Start_Date__c'
,'Responsibility__c'
,'Total_Completion_Topics__c'
,'Total_Topics__c'
,'Master_module__c'
,'RecordTypeId'
,'Workshop__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Module__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Module__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Topic__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Topic-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Actual_Complete_Date__c'
,'Actual_Start_Date__c'
,'Comments__c'
# ,'Name'
,'Planned_Complete_Date__c'
,'Planned_Start_Date__c'
,'Responsibility__c'
,'Master_Topic__c'
,'Module__c'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Topic__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Topic__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Email_Log__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Email_Log-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Email__c'
,'Email_Sent_Date__c'
,'Email1__c'
,'Email2__c'
,'First_Notification_Sent__c'
,'First_Notification_Sent_Date__c'
,'Second_Notification_Sent__c'
,'Second_Notification_Sent_Date__c'
,'Adviser__c'
,'Fee_Switch_Off_History__c'
,'Knowledge__c'
,'RecordTypeId'
,'Third_Party_Provider__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Email_Log__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Email_Log__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - CSA_Fee_Switch_Off_History__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CSA_Fee_Switch_Off_History-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Email_Sent_Status__c'
,'Email_sub_status__c'
,'Error_Message__c'
# ,'Name'
,'Processed_to_send_email__c'
,'ASF_CSA_Case__c'
,'ASF_CSA_Case_dv__c'
,'CSA_Case__c'
,'CSA_Case_dv__c'
,'Email_Log__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total CSA_Fee_Switch_Off_History__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Id'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CSA_Fee_Switch_Off_History__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - AdviserOne_Event__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_AdviserOne_Event-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Article__c'
,'Detail_URL__c'
,'Display_on_Home_Page__c'
,'Display_URL__c'
,'Event_Date__c'
,'Event_Registration_Link__c'
,'Event_Summary__c'
,'Event_Time__c'
,'Event_Type__c'
,'Home_Page_Display_Expiry__c'
,'Licensees__c'
,'Licensees_Formula__c'
# ,'Name'
,'Provider__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total AdviserOne_Event__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Id'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AdviserOne_Event__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Approval__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Approval-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Approval_Decision__c'
,'Comments__c'
,'Decision_Date__c'
,'EDP_External_Id__c'
,'Name'
,'Onboarding_Offboarding_Application__c'
,'Approver__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Approval__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Approval__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Approval__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Area_of_Advice__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Area_of_Advice-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Current_File_Number_for_Phase__c'
,'Current_Vetting_Phase__c'
,'Document_Received_Date__c'
,'EDP_External_Id__c'
# ,'Name'
,'Next_Vetting_Phase__c'
,'Number_of_Files_Approved__c'
,'Number_of_Files_in_Phase_Approved__c'
,'Number_of_Files_in_Phase_Not_Approved__c'
,'Number_of_Files_Not_Approved__c'
,'Number_of_Files_Submitted__c'
,'Reason_for_Vetting__c'
,'Vetting_Cleared_Date__c'
,'Vetting_Start_Date__c'
,'Vetting_Status__c'
,'Adviser__c'
,'Area_of_Advice__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Area_of_Advice__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Approval__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Area_of_Advice__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Breach_Reporting__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Breach_Reporting-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'A10__c'
,'A2__c'
,'A3__c'
,'A4__c'
,'A5__c'
,'A6__c'
,'A8__c'
,'ASIC_Breach_Reference__c'
,'ASIC_Lodgement_status__c'
,'ASIC_Update_Date__c'
,'BRC_Date__c'
,'BRC_For_Noting_Date__c'
,'Breach_Details__c'
,'Breach_Report_date__c'
,'Breach_Status__c'
,'Client_Loss_Investigation_Complete_Date__c'
,'Client_Loss_Investigation_Start_date__c'
,'Client_Loss_Remediation_Completed_Date__c'
,'Client_Notice_Period_start_date__c'
,'Client_Notification_sent_date__c'
,'Date_Of_The_First_Instance_Of_The_Breach__c'
,'Date_Of_The_Last_Instance_Of_The_Breach__c'
,'DNU_Breach_report_required_date__c'
,'Estimated_Completion_Of_Investigation__c'
,'External_AFSL__c'
,'Incomplete_Investigation_Status__c'
,'Investigation_Comments__c'
,'Investigation_Start_Date__c'
,'Is_the_Event_Reportable__c'
# ,'Name'
,'Name_of_subject_of_reportable_situation__c'
,'Notice_of_Client_Loss_Investigation_outc__c'
,'Number_Of_Clients_Impacted__c'
,'Number_Of_Instances_Of_The_Breach__c'
,'Number_Of_Related_Reportable_Situations__c'
,'Q1__c'
,'Q10__c'
,'Q2__c'
,'Q3__c'
,'Q4__c'
,'Q5__c'
,'Q6__c'
,'Q7__c'
,'Q8__c'
,'Q9__c'
,'Reason_For_Breach__c'
,'Remediation_Rectification_Date__c'
,'Reportable_event_confirmation_date__c'
,'Section_of_Legislation_Breached__c'
,'Sub_REF_Date__c'
,'Subject_of_reportable_situation__c'
,'Summary_BREA_Details__c'
,'Total_Financial_Loss_in_terms__c'
,'Type_Of_Breach__c'
,'Incident_File_Review__c'
,'Assigned_To__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Breach_Reporting__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Approval__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Breach_Reporting__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - CaseComment

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CaseComment-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
# 'Metazoa_External_ID__c'
'CommentBody'
,'IsPublished'
,'ParentId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total CaseComment Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CaseComment', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Breach_Reporting__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Client_File__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Client_File-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Advice_Document_Type__c'
,'Advice_Type__c'
,'Application_Date__c'
,'Chosen_for_Review__c'
,'Client_Contact_Type__c'
,'Client_File_Questionnaire_Status__c'
,'ClientID_Xplan__c'
,'Date_of_Advice__c'
,'Date_Time_of_Phone_Call__c'
,'EDP_External_Id__c'
,'Fact_Find_Date_Signed__c'
,'File_Selection_Date__c'
,'General_Advice_Score_Impact__c'
# ,'Name'
,'Number_of_General_Advice_Complaince__c'
,'Number_of_High_Issues__c'
,'Number_of_Low_Issues__c'
,'Number_of_Medium_Issues__c'
,'Number_of_Regulatory_Complaince__c'
,'Preferred_Name__c'
,'Products_Recommended__c'
,'Products_recommended_Other__c'
,'Products_Recommended_review__c'
,'Reason__c'
,'Reason_for_Choosing__c'
,'Regulatory_Compliance_Score_Impact__c'
,'Source_of_file__c'
,'Total_Number_of_Issues__c'
,'Review__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Client_File__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CaseComment', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_File__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - InitiativeChecklist__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_InitiativeChecklist-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Action_Required__c'
,'Assessment_Type__c'
,'Close_Date__c'
,'CyberSecurity_Remediation_Comment__c'
,'Due_Date__c'
,'Issue_Identified__c'
# ,'Name'
,'Practice_ID__c'
,'Priority__c'
,'Start_Date__c'
,'Status__c'
,'Initiative_Implementation__c'
,'SecurityMeasure__c'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total InitiativeChecklist__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='InitiativeChecklist__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - CommPay_Supplier__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CommPay_Supplier-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Id'
# 'Metazoa_External_ID__c'
,'CommPay_Supplier_Name__c'
,'Licensee__c'
# ,'Name'
,'Third_Party_Provider__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total CommPay_Supplier__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Complained_Advisers__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Complained_Advisers-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'EDP_External_Id__c'
# ,'Name'
,'Case__c'
,'Adviser__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Complained_Advisers__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Complained_Advisers__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - EnhancedLetterhead

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_EnhancedLetterhead-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Description'
,'LetterheadFooter'
,'LetterheadHeader'
,'Name'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total EnhancedLetterhead Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='EnhancedLetterhead', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - EmailTemplate

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_EmailTemplate-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'ApiVersion'
,'Body'
,'Description'
,'DeveloperName'
,'Encoding'
# ,'FolderName'
,'HtmlValue'
,'IsActive'
# ,'IsBuilderContent'
# ,'LastUsedDate'
,'Markup'
,'Name'
# ,'NamespacePrefix'
,'RelatedEntityType'
,'Subject'
,'TemplateStyle'
,'TemplateType'
# ,'TimesUsed'
,'UiType'
,'BrandTemplateId'
,'EnhancedLetterheadId'
,'FolderId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total EmailTemplate Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='EmailTemplate', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Consequence_Management_Forum_Entry__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Consequence_Management_Forum_Entry-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Client_s_impacted__c'
# ,'CMF__c'
,'Consequences__c'
,'Description__c'
,'EDP_External_Id__c'
,'Explanation_to_support_Consequences__c'
,'Meeting_date__c'
# ,'Name'
,'Nature_of_client_detriment__c'
,'Potential_dollar_loss_or_detriment__c'
,'Received_Date__c'
,'Recommendation__c'
,'Root_Cause_s__c'
,'Significant_Issue_Patterns__c'
,'Source_of_Submission__c'
,'Practice__c'
,'Assurance_Review__c'
,'Adviser__c'
,'RecordTypeId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Consequence_Management_Forum_Entry__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Consequence_Management_Forum_Entry__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Event

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Event-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'ActivityDate'
,'ActivityDateTime'
,'AdviceAssuranceTeamComments__c'
,'ASN_Decision__c'
,'Changed_By__c'
,'Comments__c'
,'Communication_Channel__c'
,'Completed_Date__c'
,'Contact_Date__c'
,'Continue_to_Next_Phase__c'
,'Created_from_Std_QA_batch__c'
,'Date_registered_booked_sent__c'
,'Description'
,'DurationInMinutes'
,'EDP_External_Id__c'
# ,'EndDate'
,'EndDateTime'
,'EventSubtype'
,'Future_time_period_Month_Year__c'
# ,'GroupEventType'
,'IsAllDayEvent'
# ,'IsArchived'
# ,'IsChild'
# ,'IsGroupEvent'
,'IsPrivate'
,'IsRecurrence'
# ,'IsRecurrence2'
,'IsRecurrence2Exception'
# ,'IsRecurrence2Exclusion'
,'IsReminderSet'
,'IsVisibleInSelfService'
,'Legal_Privileged_Confidential__c'
,'Licensee_Supervisor_Comments__c'
,'Location'
,'Preferred_Name__c'
,'Reason_Why__c'
,'Received_completed_attended_Date__c'
# ,'Recurrence2PatternStartDate'
,'Recurrence2PatternText'
# ,'Recurrence2PatternTimeZone'
# ,'Recurrence2PatternVersion'
,'RecurrenceDayOfMonth'
,'RecurrenceDayOfWeekMask'
,'RecurrenceEndDateOnly'
,'RecurrenceInstance'
,'RecurrenceInterval'
,'RecurrenceMonthOfYear'
,'RecurrenceStartDateTime'
,'RecurrenceTimeZoneSidKey'
,'RecurrenceType'
,'ReminderDateTime'
,'Resource_Hours__c'
,'Section__c'
,'ShowAs'
,'Source_Document__c'
,'StartDateTime'
,'Sub_Task__c'
,'Subject'
,'Support_Staff_Comments__c'
# ,'Task_Number__c'
,'Task_Owner__c'
,'Technical_Proficiency_Rating__c'
,'Type'
# ,'WhatCount'
# ,'WhoCount'
,'WhatId'
# ,'AccountId'
,'ASN_Indicator_History__c'
,'Assurance_Review__c'
,'WhoId'
# ,'RecurrenceActivityId'
,'Question_Response__c'
,'Response_Finding__c'
,'Colleague__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Event Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Event', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Event', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - EventRelation

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_EventRelation-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
# 'Metazoa_External_ID__c'
'IsInvitee'
,'IsParent'
,'IsWhat'
,'RespondedDate'
,'Response'
,'Status'
,'RelationId'
# ,'AccountId'
,'EventId'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total EventRelation Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='EventRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Event', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - NAPW__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_NAPW-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'advice_for__c'
,'adviser_id__c'
,'Advisername__c'
,'AdviserSFID__c'
,'area_of_advice__c'
,'ASIC_AR_Number__c'
,'best_interest__c'
,'client_adviser__c'
,'client_entity_name__c'
,'client_id__c'
,'distribution_channel__c'
,'expiry_date__c'
,'Firstname__c'
,'fum__c'
,'individual_or_group_cover__c'
,'inside_or_outside_super__c'
,'Lastname__c'
,'life_sum_insured__c'
# ,'Name'
,'napw_apir_asx_code__c'
,'napw_external_research__c'
,'new_or_existing__c'
,'outcome__c'
,'outcome_commentary__c'
,'outcome_date__c'
,'partner_entity_name__c'
,'partner_id__c'
,'primary_group__c'
,'product_name__c'
,'salary_continuance_sum_insured__c'
,'submission_date__c'
,'tmd_link__c'
,'tpd_sum_insured__c'
,'trauma_sum_insured__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total NAPW__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Qualification_Type__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Qualification_Type-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'EDP_External_Id__c'
,'Name'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Qualification_Type__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Qualification_Type__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Qualification_Type__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Qualification__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Qualification-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'EDP_External_Id__c'
# ,'Name'
,'Other_Qualification__c'
,'Other_Registered_Training_Organisation__c'
,'Qualification_Course_Status__c'
,'Qualification_TypeNew__c'
,'Specialist_Accreditation__c'
,'Subject_Name__c'
,'TASA__c'
,'Year_Obtained__c'
,'Adviser__c'
,'Qualification_Type__c'
,'Registered_Training_Organisation__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Qualification__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Qualification__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Response_Finding__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Response_Finding-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Actions_Taken__c'
,'ASN_Indicator__c'
,'Due_Date__c'
,'Findings__c'
# ,'Name'
,'Root_Cause__c'
,'Master_Question__c'
,'MasterQuestions_Targetedreviews__c'
,'Incident__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Response_Finding__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Response_Finding__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - SecondaryPracticeAdviserHistory__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_SecondaryPracticeAdviserHistory-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'EDP_External_Id__c'
,'End_Date__c'
# ,'Name'
,'Roles__c'
,'Start_Date__c'
,'Practice__c'
,'Adviser__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total SecondaryPracticeAdviserHistory__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='SecondaryPracticeAdviserHistory__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - SelectedQuestions_Targetedreviews__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_SelectedQuestions_Targetedreviews-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Cloned__c'
# ,'Name'
,'Question_Number__c'
,'Question_Text__c'
,'Sorting_Index__c'
,'Assurance_Review__c'
,'MasterQuestions_Targetedreviews__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total SelectedQuestions_Targetedreviews__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='SelectedQuestions_Targetedreviews__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Snapshot_Data__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Snapshot_Data-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Case_Status__c'
,'Case_Type__c'
,'EDP_External_Id__c'
# ,'Name'
,'Shapshot_Frequency__c'
,'Snapshot_Date__c'
,'Snapshot_Type__c'
,'Status__c'
,'Vetting_Reason__c'
,'Vetting_Status__c'
,'Area_of_Advice__c'
,'Case__c'
,'Incident__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Snapshot_Data__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Snapshot_Data__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - ASN_Indicator_History__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_ASN_Indicator_History-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'ASN__c'
# ,'Name'
,'Review_Count_or_Average__c'
,'Type__c'
,'Assurance_Review__c'
,'Adviser__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total ASN_Indicator_History__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='TaskRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='ASN_Indicator_History__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Reviews_Accounted__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Reviews_Accounted-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
'Metazoa_External_ID__c'
,'Indicator_Score__c'
# ,'Name'
,'ASN_Indicator_History__c'
,'Assurance_Review__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Reviews_Accounted__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='TaskRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Reviews_Accounted__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Task

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'ActivityDate'
,'AdviceAssuranceTeamComments__c'
,'ASN_Decision__c'
,'CallDisposition'
,'CallDurationInSeconds'
,'CallObject'
,'CallType'
,'Changed_By__c'
,'Comments__c'
,'Communication_Channel__c'
,'Completed_Date__c'
# ,'CompletedDateTime'
,'Contact_Date__c'
,'Continue_to_Next_Phase__c'
,'Created_from_Std_QA_batch__c'
,'Date_registered_booked_sent__c'
,'Description'
,'EDP_External_Id__c'
,'Future_time_period_Month_Year__c'
# ,'IsArchived'
# ,'IsClosed'
# ,'IsHighPriority'
# ,'IsRecurrence'
,'IsReminderSet'
,'IsVisibleInSelfService'
,'Legal_Privileged_Confidential__c'
,'Licensee_Supervisor_Comments__c'
,'Preferred_Name__c'
,'Priority'
,'Reason_Why__c'
,'Received_completed_attended_Date__c'
,'RecurrenceDayOfMonth'
,'RecurrenceDayOfWeekMask'
,'RecurrenceEndDateOnly'
,'RecurrenceInstance'
,'RecurrenceInterval'
,'RecurrenceMonthOfYear'
,'RecurrenceRegeneratedType'
,'RecurrenceStartDateOnly'
,'RecurrenceTimeZoneSidKey'
,'RecurrenceType'
,'ReminderDateTime'
,'Resource_Hours__c'
,'Section__c'
,'Source_Document__c'
,'Status'
,'Sub_Task__c'
,'Subject'
,'Support_Staff_Comments__c'
# ,'Task_Number__c'
,'Task_Owner__c'
# ,'TaskSubtype'
,'Technical_Proficiency_Rating__c'
,'Type'
# ,'WhatCount'
# ,'WhoCount'
,'WhatId'
# ,'AccountId'
,'ASN_Indicator_History__c'
,'Assurance_Review__c'
,'WhoId'
,'Question_Response__c'
,'RecordTypeId'
,'Response_Finding__c'
# ,'RecurrenceActivityId'
,'Colleague__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Task Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='TaskRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Task', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=2000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Vetting_Result__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Vetting_Result-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'ARResubmitted__c'
,'EDP_External_Id__c'
,'File_Number__c'
,'MandatoryQuestionsFail__c'
# ,'Name'
,'Number_of_High_Issues__c'
,'Number_of_Low_Issues__c'
,'Number_of_Medium_issues__c'
,'Phase_Number__c'
,'Reason_for_Vetting__c'
,'Total_Number_of_Issues__c'
,'Vetting_Status_at_Finish__c'
,'Vetting_Status_at_Start__c'
,'Area_of_Advice__c'
,'Assurance_Review__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Vetting_Result__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='TaskRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Vetting_Result__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Wealth_Central_Adoption__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Wealth_Central_Adoption-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
'Metazoa_External_ID__c'
,'Access_before_1_Oct_2021__c'
,'Access_before_1_Oct_2021_C__c'
,'Adviser_Moved_Practice__c'
,'Advocate__c'
,'Comments__c'
,'Comments_Non_HTML__c'
,'Committing_to_Adoption_Program__c'
,'Good_candidate_for_WC__c'
,'Historic_IOOF_pilot_participant__c'
,'Level_of_WC_access_unlocked__c'
,'MLC_Pilot_participant__c'
# ,'Name'
,'Not_interested_ready__c'
,'Peer_Group__c'
,'Reason_Why__c'
,'Successfully_embedded_WC__c'
,'Terminated_access__c'
,'Practice__c'
,'Adviser__c'
,'NominatedChampionAdviser__c'
,'RecordTypeId'
,'User__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Wealth_Central_Adoption__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='TaskRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Wealth_Central_Adoption__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Adviser_ASN_Snapshot__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Adviser_ASN_Snapshot-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
# 'Metazoa_External_ID__c'
'Adviser_ASN__c'
,'ASN_Source__c'
,'ASN_Start_Date__c'
,'Execution_Time__c'
# ,'Name'
,'Adviser__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Adviser_ASN_Snapshot__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Adviser_ASN_Snapshot__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Adviser_ASN_Snapshot__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Refund_Request__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Refund_Request-ALL]
;"""

In [ ]:
ls_tgt_objFields= [
# 'Id'
# 'Metazoa_External_ID__c'
'Alternative_payment_Confirmation_Date__c'
,'Alternative_payment_Request_Sent_Date__c'
,'At_Fault__c'
,'Calculation_required__c'
,'Client_Category__c'
,'Comments__c'
,'Confirm_payment_with_Trustee__c'
,'Counterfactual__c'
,'Date_Refund_Assessed__c'
,'Date_Refund_Processed__c'
,'Date_Refund_Submitted__c'
,'Final_Refund_Amount__c'
,'Form_Submitted_Date__c'
,'Further_Information_provided__c'
,'Further_information_received__c'
,'Further_Information_required__c'
,'Have_you_provided_bank_details__c'
,'Interest__c'
,'LICENSEE_NAME__c'
# ,'Name'
,'Name_of_Client_s__c'
,'Payment_confirmation_received__c'
,'Payment_processing_method__c'
,'Payment_request_lodged__c'
,'Policy_Number__c'
,'Policy_Status__c'
,'Product_Platform_s_For_Refund_Payment__c'
,'Product_Provider__c'
,'Reason_for_Refund__c'
,'Refund_In_Scope__c'
,'Refund_Period_End_Date__c'
,'Refund_Period_Start_Date__c'
,'Refund_Required__c'
,'Refund_Type__c'
,'Related_Entity__c'
,'Remediation_Refund__c'
,'Request_Sent_Date__c'
,'Revenue_Confirmation_Date__c'
,'Stage__c'
,'Status__c'
,'Status_Alt_Payment__c'
,'Status_Revenue__c'
,'Trustee_confirmation_of_Payment__c'
,'Trustee_manager_notification__c'
,'Trustee_outcome__c'
,'Trustee_Process_Start_Date__c'
,'Type_of_Payment__c'
,'Xplan_ID__c'
,'Case_Number__c'
,'Adviser_Name__c'
,'Incident_Form__c'
,'RecordTypeId'
,'Assigned_To_Ops_Team__c'
,'Assigned_to_Trustee_Payment__c'
,'Responsible__c'
]

In [ ]:
engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

In [ ]:
df_transformed

In [ ]:
logger.info(f"Total Refund_Request__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Refund_Request__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Refund_Request__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")